In [0]:
%run "../00_common/01_environment_config"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - BUILD RACES DIMENSION
# ============================================================
# Purpose:
# Joins Silver races and circuits data to create an
# analytics-ready races dimension in the Gold layer.
# ============================================================


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

races_path = f"{s3_root_path}/Silver/Kalashri/races"

circuits_path = f"{s3_root_path}/Silver/Kalashri/circuits"

target_path = f"{s3_root_path}/Gold/Kalashri/dim_races"

print("Races Path    :", races_path)
print("Circuits Path :", circuits_path)
print("Target Path   :", target_path)


# COMMAND ----------

# ============================================================
# STEP 1 - READ SILVER DATA
# ============================================================

races_df = (
    spark.read
        .format("delta")
        .load(races_path)
)

circuits_df = (
    spark.read
        .format("delta")
        .load(circuits_path)
)

display(races_df.limit(20))
display(circuits_df.limit(20))


# COMMAND ----------

# Check schemas before join

races_df.printSchema()
circuits_df.printSchema()


# COMMAND ----------

# ============================================================
# STEP 2 - JOIN RACES WITH CIRCUITS
# ============================================================

dim_races_df = (
    races_df
        .join(
            circuits_df,
            races_df.circuit_id == circuits_df.circuit_id,
            "inner"
        )
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.date.alias("race_date"),
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

display(dim_races_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 3 - WRITE TO GOLD
# ============================================================

(
    dim_races_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
)


# COMMAND ----------

# ============================================================
# STEP 4 - VALIDATE GOLD DATA
# ============================================================

gold_dim_races_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_dim_races_df.limit(20))

print("Gold Records :", gold_dim_races_df.count())

print("Gold dim_races successfully written to:")
print(target_path)

In [0]:
%sql

SELECT *
FROM delta.`s3://bizf1project123/Gold/Kalashri/dim_races`;